In [ ]:
#RECUPERATION EQUIPES INRIA

import pandas as pd
import requests
import pyarrow as pa



######
# Récupération des acronymes des équipes de l'API HAL
# https://api.archives-ouvertes.fr/ref/structure/type_s:researchteam AND (parentDocid_i:419153 OR parentDocid_i:104751 OR parentDocid_i:34586 OR parentDocid_i:2497 OR parentDocid_i:1096051 OR parentDocid_i:129671 OR parentDocid_i:104752 OR parentDocid_i:118511 OR parentDocid_i:454310)
url = "https://api.archives-ouvertes.fr/ref/structure/"
params = {
    "q": "type_s:researchteam AND (parentDocid_i:419153 OR parentDocid_i:104751 OR parentDocid_i:34586 OR parentDocid_i:2497 OR parentDocid_i:1096051 OR parentDocid_i:129671 OR parentDocid_i:104752 OR parentDocid_i:118511 OR parentDocid_i:454310 OR parentDocid_i:1175218 OR parentDocid_i:1225635 OR parentDocid_i:1225627)",
    "fl": "parentName_s,parentDocid_i,acronym_s,docid,valid_s,startDate_s,endDate_s",
    "rows": 1000,
    "wt": "json",
}
response = requests.get(url, params=params)
docid_acronyme_dict = {}
docid_parentname_dict = {}
if response.status_code == 200:
    results_json = response.json()
    docs = results_json['response']['docs']

    data = [] # création d'une liste 

    for doc in docs:
        Datedebut = doc.get('startDate_s', '9999')
        Datedebut = Datedebut[0][:4] #4 premiers caractères
        Datefin = doc.get('endDate_s', '9999')
        Datefin = Datefin[0][:4]  #4 premiers caractères
        validity = doc['valid_s']
        docid = doc['docid']
        acronyme = doc.get('acronym_s', 'N/A')
        parent_docids = doc.get('parentDocid_i', [])
        parent_names = []  # Liste pour stocker les valeurs de parent_name
        for parent_docid in parent_docids:
            # Affecter une valeur différente en fonction du parent_docid
            if parent_docid == '419153':
                parent_name = 'Centre Inria univ. Rennes '
                parent_names.append(parent_name)
            elif parent_docid == '104751':
                parent_name = 'Centre Inria univ. Bordeaux'
                parent_names.append(parent_name)
            elif parent_docid == '34586':
                parent_name = 'Centre Inria univ. Côte Azur'
                parent_names.append(parent_name)
            elif parent_docid == '2497':
                parent_name = 'Centre Inria univ. Grenoble Alpes'
                parent_names.append(parent_name)
            elif parent_docid == '1096051':
                parent_name = 'Centre Inria de Lyon'
                parent_names.append(parent_name)
            elif parent_docid == '129671':
                parent_name = 'Centre Inria univ. Lorraine'
                parent_names.append(parent_name)
            elif parent_docid == '104752':
                parent_name = 'Centre Inria de Lille'
                parent_names.append(parent_name)
            elif parent_docid == '118511':
                parent_name = 'Centre Inria de Saclay'
                parent_names.append(parent_name)
            elif parent_docid == '454310':
                parent_name = 'Centre Inria de Paris'
                parent_names.append(parent_name)
            elif parent_docid == '1175218':
                parent_name = 'Centre Inria de Paris'
                parent_names.append(parent_name)
            elif parent_docid == '1225635' :
                parent_name = 'Centre Inria de Paris'
                parent_names.append(parent_name)
            elif  parent_docid == '1225627':
                parent_name = 'Centre Inria de Saclay'
                parent_names.append(parent_name)
        #Condition générale pour les cas non spécifiés
        if parent_names:
        # Concaténer les valeurs de parent_name si la liste n'est pas vide
            parent_name = ', '.join(parent_names)
        else:
            parent_name = docid
        # docid_acronyme_dict[docid] = acronyme
        # docid_parentname_dict[docid] = parent_name
        data.append([validity,docid, acronyme, parent_name,Datedebut,Datefin])

    # Créer un DataFrame pandas à partir des données
    df_teams = pd.DataFrame(data, columns=['validity','docid', 'acronyme', 'parent_name','Date_debut','Date_fin'])
    
    # Ajout de lignes manquantes dans le DataFrame
    additional_data = [
        ['VALID',419153, 'Centre Inria univ. Rennes', 'Centre Inria univ. Rennes','',''], ['VALID',104751, 'Centre Inria univ. Bordeaux', 'Centre Inria univ. Bordeaux','',''],
        ['VALID',34586, 'Centre Inria univ. Côte Azur', 'Centre Inria univ. Côte Azur','',''], ['VALID',2497, 'Centre Inria univ. Grenoble Alpes', 'Centre Inria univ. Grenoble Alpes',''],
        ['VALID',1096051, 'Centre Inria de Lyon', 'Centre Inria de Lyon','',''], ['VALID',129671, 'Centre Inria univ. Lorraine', 'Centre Inria univ. Lorraine','',''],
        ['VALID',104752, 'Centre Inria de Lille', 'Centre Inria de Lille','',''], ['VALID',118511, 'Centre Inria de Saclay', 'Centre Inria de Saclay','',''],
        ['VALID',454310, 'Centre Inria de Paris', 'Centre Inria de Paris','','']
    ]
    df_teams = pd.concat([df_teams, pd.DataFrame(additional_data, columns=df_teams.columns)], ignore_index=True)



print(f"Première partie terminée (nous avons récupéré les équipes Inria dans Auréhal)")

# Conversion explicite des types de données
df_teams['docid'] = df_teams['docid'].astype(str)
df_teams['acronyme'] = df_teams['acronyme'].astype(str)
df_teams['parent_name'] = df_teams['parent_name'].astype(str)
df_teams['validity'] = df_teams['validity'].astype(str)
df_teams['Date_debut'] = df_teams['Date_debut'].astype(str)
df_teams['Date_fin'] = df_teams['Date_fin'].astype(str)


df_teams.to_excel("equipes_Inria.xlsx", index=False)

print ("Fichier des equipes Inria généré")

In [ ]:
Identification dans HAL des projets ANR où Inria publie

In [ ]:
import requests

def fetch_anr_project_references():
    base_url = "https://api.archives-ouvertes.fr/search/INRIA/"
    cursorMark = "*"
    all_anr_refs = set()  # Utilisation d'un ensemble pour éviter les doublons

    while True:
        params = {
            "q": "anrProjectReference_s:*",
            "fq": "publicationDateY_i:[2020 TO 2025]",
            "rows": 500,
            "sort": "docid asc",
            "cursorMark": cursorMark,
            "fl": "anrProjectReference_s"
        }

        response = requests.get(base_url, params=params)
        data = response.json()

        docs = data.get("response", {}).get("docs", [])
        if not docs:
            break

        # Extraction des références ANR
        for doc in docs:
            anr_refs = doc.get("anrProjectReference_s", [])
            for ref in anr_refs:
                all_anr_refs.add(ref)

        next_cursorMark = data.get("nextCursorMark", cursorMark)
        if next_cursorMark == cursorMark:
            break

        cursorMark = next_cursorMark

    return sorted(all_anr_refs)  # Retourne la liste triée des références uniques

# Exécution
anr_project_references = fetch_anr_project_references()
print(f"Nombre de références ANR uniques : {len(anr_project_references)}")
# print("Liste des références :")
# for ref in anr_project_references:
#     print(ref)
#Export Excel pour la liste

#  `anr_project_references` est une liste de références ANR
anr_project_references = fetch_anr_project_references()  

# Création du DataFrame avec une seule colonne
inria_in_anr_projects = pd.DataFrame(anr_project_references, columns=["anrProjectReference_s"])

# Export vers Excel
inria_in_anr_projects.to_excel("inria_in_anr_projects.xlsx", index=False)

In [ ]:
# Extraction de toutes les publications et des affiliations à partir de la liste des projets ANR où Inria publie
import requests
import pandas as pd
import os

# =========================
# 📁 CONFIG
# =========================
INPUT_FILE = "inria_in_anr_projects.xlsx"
OUTPUT_DIR = "ANR_Hal_only"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 🌐 FETCH HAL
# =========================
def fetch_hal_data(anr_ref):
    url = "https://api.archives-ouvertes.fr/search/"
    cursor = "*"
    all_docs = []

    params = {
        "q": f'anrProjectReference_s:"{anr_ref}"',
        "fq":"publicationDateY_i:[2020 TO 2025]",
        "rows": 500,
        "sort": "docid asc",
        "cursorMark": cursor,
        "fl": "anrProjectReference_s,anrProjectAcronym_s,halId_s,structIdName_fs,collCode_s,publicationDateY_i"
    }

    while True:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()

        docs = data.get("response", {}).get("docs", [])
        if not docs:
            break

        all_docs.extend(docs)

        next_cursor = data.get("nextCursorMark")
        if next_cursor == cursor:
            break

        params["cursorMark"] = next_cursor
        cursor = next_cursor

    return all_docs

# =========================
# 🧼 CLEAN
# =========================
def clean_docs(docs, anr_ref):
    rows = []

    for d in docs:
        hal_id = d.get("halId_s")
        year = d.get("publicationDateY_i")
        coll_code = d.get("collCode_s", [])

        # 🔧 collCode aplati une fois
        if isinstance(coll_code, list):
            coll_code = " | ".join(coll_code)

        struct_entities = d.get("structIdName_fs", [])

        if not struct_entities:
            # fallback si aucune structure
            rows.append({
                "anrProjectReference": anr_ref,
                "halId": hal_id,
                "structCode": "UNKNOWN",
                "structName": "UNKNOWN",
                "collCode": coll_code,
                "year": year
            })
            continue

        for s in struct_entities:
            if "_FacetSep_" not in s:
                continue

            parts = s.split("_FacetSep_")
            if len(parts) < 2:
                continue

            code = parts[0].strip()
            name = parts[1].strip()

            rows.append({
                "anrProjectReference": anr_ref,
                "halId": hal_id,
                "structCode": code,
                "structName": name,
                "collCode": coll_code,
                "year": year
            })

    df = pd.DataFrame(rows)

    if df.empty:
        return df
    
    # 🔎 supprimer les lignes où structCode = 300009 sans mention INRIA dans collCode
    mask = (df["structCode"] == "300009") & (
        ~df["collCode"].fillna("").str.contains("INRIA", case=False)
    )

    df = df[~mask]

    # nettoyage
    df = df.drop_duplicates()
    df = df.sort_values(["year", "halId"])

    return df

 

# =========================
# 💾 EXPORT
# =========================
def save_to_excel(anr_ref, df):
    if df.empty:
        print(f"Aucune donnée pour {anr_ref}")
        return

    filename = os.path.join(OUTPUT_DIR, f"{anr_ref}.xlsx")
    df.to_excel(filename, index=False)

    print(f"✔ {anr_ref} → {len(df)} lignes")

# =========================
# 🚀 MAIN
# =========================
if __name__ == "__main__":

    df_input = pd.read_excel(INPUT_FILE)
    anr_refs = df_input["anrProjectReference_s"].dropna().unique()

    volume_rows = []  # 👈 pour le résumé

    for anr_ref in anr_refs:
        try:
            docs = fetch_hal_data(anr_ref)
            print (f"nombre de docs pour {anr_ref} : {len(docs)}")

            # 📊 stock volume brut
            volume_rows.append({
                "anrProjectReference": anr_ref,
                "nb_publications": len(docs)
            })

            df_clean = clean_docs(docs, anr_ref)
            save_to_excel(anr_ref, df_clean)

        except Exception as e:
            print(f"❌ Erreur pour {anr_ref} : {e}")

    # =========================
    # 📊 EXPORT VOLUME GLOBAL
    # =========================
    df_volume = pd.DataFrame(volume_rows)

    if not df_volume.empty:
        df_volume = df_volume.sort_values("nb_publications", ascending=False)

        output_path = os.path.join(OUTPUT_DIR, "volume_publis_anr.xlsx")
        df_volume.to_excel(output_path, index=False)

        print(f"📊 Fichier volume généré : {output_path}")

    print("🌙 Terminé proprement")

Graphiques

Préparation du dataset pour générer les graphiques

In [ ]:
import os
import pandas as pd
import unicodedata

# =========================
# 📁 PARAMÈTRES
# =========================
FOLDER_PATH = "ANR_Hal_only"
OUTPUT_FILE = "df_hal_global.parquet"

# =========================
# 🧼 CLEAN
# =========================
def clean_text(x):
    return unicodedata.normalize("NFKC", str(x).strip())

# =========================
# 📥 CHARGEMENT
# =========================
dfs = []

for f in os.listdir(FOLDER_PATH):
    if not f.endswith(".xlsx"):
        continue

    df_tmp = pd.read_excel(os.path.join(FOLDER_PATH, f))
    dfs.append(df_tmp)
    print(f"{f} traité")

df = pd.concat(dfs, ignore_index=True)

# =========================
# 🧼 NORMALISATION
# =========================
df["structName"] = df["structName"].apply(clean_text)
df["structCode"] = df["structCode"].astype(str).str.strip()
df["year"] = pd.to_numeric(df["year"], errors="coerce")

df = df.dropna(subset=["structName", "halId", "year"])

# =========================
# 🔑 CLÉ D'ANALYSE = structName
# =========================
df = df.drop_duplicates(
    subset=["anrProjectReference", "halId", "structName", "year"]
)

# =========================
# 💾 EXPORT
# =========================
df.to_parquet(OUTPUT_FILE, index=False)

print("✅ Dataset global construit :", OUTPUT_FILE)

In [ ]:
### Graphique ULTIMATE

import os
import re
import pandas as pd
import plotly.graph_objects as go
import unicodedata

# =======================
# ⚙️ PARAMÈTRES
# =======================
PROJECT_FILE = "inria_in_anr_projects.xlsx"
FOLDER_PATH = "ANR_Hal_only"
OUTPUT_FOLDER = "ANR_Hal_only/graphiques_hal"
NB_PUBLIS_FILE = "ANR_HAL_Nb_Publis_par_projets_2020_2025.xlsx"



YEAR_MIN = 2020
YEAR_MAX = 2025

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
# =======================
# 📥 nb de publis
# =======================
df_nb = pd.read_excel(NB_PUBLIS_FILE)
df_nb.columns = df_nb.columns.str.strip()

print(df_nb.columns.tolist())
nb_map = dict(zip(
    df_nb["anrProjectReference"],
    df_nb["Nb publications"]
))

# =======================
# 📥 LISTE ANR OFFICIELLE
# =======================
projects = pd.read_excel(PROJECT_FILE)
anr_list = set(projects["anrProjectReference_s"].dropna().unique())

# =======================
# 📥 CHARGEMENT HAL
# =======================
dfs = []

for f in os.listdir(FOLDER_PATH):
    if not f.endswith(".xlsx"):
        continue

    anr_code = f.replace(".xlsx", "")

    # if anr_code not in anr_list:
    #     continue

    df_tmp = pd.read_excel(
        os.path.join(FOLDER_PATH, f),
        usecols=[
            "anrProjectReference",
            "structCode",
            "structName",
            "halId",
            "year"
        ]
    )

    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# =======================
# 🧼 CLEAN
# =======================
def clean(x):
    return unicodedata.normalize("NFKC", str(x).strip())


def normalize_string(s):
    """Normalise une chaîne en supprimant les accents et en mettant en minuscules."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize('NFKD', s)
    s = ''.join([c for c in s if not unicodedata.combining(c)]).lower()
    return s

def clean_struct_name(name):
    """Nettoie le nom de la structure et remplace le nom complet de l'Inria par 'Inria'."""
    if not isinstance(name, str):
        return name

    name = name.strip()
    normalized_name = normalize_string(name)
    inria_full_name = normalize_string("institut national de recherche en informatique et en automatique")

    if normalized_name == inria_full_name:
        return "Inria"

    return name

def contains_inria(name):
    """Vérifie si le nom contient 'inria' (insensible à la casse et aux accents)."""
    if not isinstance(name, str):
        return False
    return "inria" in normalize_string(name)

def color(name):
    
    return "red" if contains_inria(name) else "#1f77b4"


df["structName"] = df["structName"].apply(clean)
df["structName"] = df["structName"].apply(clean_struct_name)
df["structCode"] = df["structCode"].astype(str).str.strip()
df["year"] = pd.to_numeric(df["year"], errors="coerce")

df = df.dropna(subset=["structCode", "structName", "halId", "year"])

df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)]

# =======================
# 🗺️ MAP NOMS
# =======================
name_map = (
    df.drop_duplicates("structCode")
    .set_index("structCode")["structName"]
    .to_dict()
)

# =======================
# 🧠 TRI ANR (25 → 15)
# =======================
def extract_year(anr):
    m = re.search(r"ANR-(\d+)", anr)
    return int(m.group(1)) if m else -1

anr_list = sorted(anr_list, key=extract_year, reverse=True)

# =========================================================
# 🌍 GLOBAL
# =========================================================
global_unique = df.drop_duplicates(subset=["structName", "halId"])

top_global = (
    global_unique.groupby("structName")["halId"]
    .count()
    .sort_values(ascending=False)
    .head(20)
    .index
)

global_unique = global_unique[global_unique["structName"].isin(top_global)]

global_agg = (
    global_unique
    .groupby(["structName", "year"])["halId"]
    .count()
    .reset_index()
)

order_global = (
    global_agg.groupby("structName")["halId"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig = go.Figure()

for y in sorted(global_agg["year"].unique()):
    sub = global_agg[global_agg["year"] == y]

    g = sub.set_index("structName")["halId"].reindex(order_global, fill_value=0)

    names = [name_map.get(c, c) for c in g.index]

    fig.add_trace(go.Bar(
        x=names,
        y=g.values,
        name=str(y),
        marker=dict(color=[color(n) for n in names])
    ))

fig.update_layout(
    title="GLOBAL HAL - Top 20 structures",
    barmode="stack",
    height=800
)

fig.write_html(os.path.join(OUTPUT_FOLDER, "ALL.html"), include_plotlyjs="cdn")

# =========================================================
# 🟦 ANR GRAPHS
# =========================================================
# df_inria = df[
#     df["structName"].astype(str).str.strip().str.lower()
#     == "institut national de recherche en informatique et en automatique"
# ].copy()

df_inria = df[df["structName"] == "Inria"].copy()

inria_unique = df_inria.drop_duplicates(subset=["anrProjectReference", "halId"])

inria_counts = (
    inria_unique.groupby("anrProjectReference")["halId"]
    .nunique()
    .reset_index()
    .rename(columns={"halId": "inria_count"})
)

for anr in anr_list:

    df_anr = df[df["anrProjectReference"] == anr].copy()
    if df_anr.empty:
        continue

    df_unique = df_anr.drop_duplicates(subset=["structCode", "halId"])

    top20 = (
        df_unique.groupby("structCode")["halId"]
        .count()
        .sort_values(ascending=False)
        .head(20)
        .index
    )

    df_unique = df_unique[df_unique["structCode"].isin(top20)]

    order = (
        df_unique.groupby("structCode")["halId"]
        .count()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    agg = (
        df_unique
        .groupby(["structCode", "year"])["halId"]
        .count()
        .reset_index()
    )

    fig = go.Figure()

    for y in sorted(agg["year"].unique()):
        sub = agg[agg["year"] == y]

        g = sub.set_index("structCode")["halId"].reindex(order, fill_value=0)

        names = [name_map.get(c, c) for c in g.index]

        fig.add_trace(go.Bar(
            x=names,
            y=g.values,
            name=str(y),
            marker=dict(color=[color(n) for n in names])
        ))

    fig.update_layout(
        title=f"{anr} - Top 20 structures HAL",
        barmode="stack",
        height=800
    )

    fig.write_html(os.path.join(OUTPUT_FOLDER, f"{anr}.html"), include_plotlyjs="cdn")

# =========================================================
# 🔥 HEATMAP INRIA (Top 30 projets avec le plus de publications Inria)
# =========================================================

# Calculer le nombre total de publications par projet ANR
total_anr = (
    df.drop_duplicates(subset=["anrProjectReference", "halId"])
    .groupby("anrProjectReference")["halId"]
    .nunique()
    .reset_index()
    .rename(columns={"halId": "total"})
)

# Fusionner avec les comptes Inria
heat = inria_counts.merge(total_anr, on="anrProjectReference", how="left")

# Calculer le ratio
heat["ratio"] = heat["inria_count"] / heat["total"]

# Trier par nombre de publications Inria (décroissant) et garder les 20 premiers
heat = heat.sort_values("inria_count", ascending=False).head(30)

# Créer le graphique
fig_heat = go.Figure()

fig_heat.add_trace(go.Bar(
    x=heat["anrProjectReference"],
    y=heat["inria_count"],
    marker=dict(color=heat["inria_count"], colorscale="Reds"),
    text=heat["inria_count"],  # Ajouter les valeurs comme texte sur les barres
    textposition='auto',      # Position automatique du texte
))

fig_heat.update_layout(
    title="Top 30 projets ANR avec le plus de publications Inria",
    xaxis=dict(tickangle=-45),
    yaxis_title="Nombre de publications Inria",
    height=600
)



fig_heat.write_html(os.path.join(OUTPUT_FOLDER, "INRIA_heatmap.html"), include_plotlyjs="cdn")
print(f"Heatmap Inria (Top 20) générée : {OUTPUT_FOLDER}/INRIA_heatmap.html")

# =========================================================
# 🌐 DASHBOARD HTML
# =========================================================

html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>ANR - Participation Inria source HAL</title>

<style>
body { display:flex; margin:0; font-family:Arial; }

.sidebar {
    width:320px;
    height:100vh;
    overflow-y:auto;
    background:#f5f5f5;
    padding:10px;
}

button { width:100%; padding:6px; margin:2px 0; }

iframe { flex:1; height:100vh; border:none; }
</style>

<script>
function loadGraph(f){ document.getElementById("frame").src = f; }

function filterANR(){
 let i=document.getElementById("search").value.toLowerCase();
 for(let b of document.getElementsByClassName("anr")){
  b.style.display=b.innerText.toLowerCase().includes(i)?"block":"none";
 }
}
</script>

</head>
<body>

<div class="sidebar">

<h3>🌍 Global</h3>
<button onclick="loadGraph('ALL.html')">Contributeurs projets ANR</button>

<h3>🔥 Inria</h3>
<button onclick="loadGraph('INRIA_heatmap.html')">Top 30 contributions Inria</button>

<h3>🔎 ANR</h3>
<input id="search" onkeyup="filterANR()" placeholder="ANR..." style="width:100%;padding:6px">

"""

for anr in anr_list:
    nb = nb_map.get(anr, 0)
    html += f'<button class="anr" onclick="loadGraph(\'{anr}.html\')">{anr} ({nb})</button>'


html += """
</div>

<iframe id="frame" src="ALL.html"></iframe>

</body>
</html>
"""

with open(os.path.join(OUTPUT_FOLDER, "index.html"), "w", encoding="utf-8") as f:
    f.write(html)

print("✅ Dashboard complet généré")